# Semantic search and result display

This notebook only performs retrieval. Chunks are searched internally, results are grouped by canonical article ID, and the complete legal article is displayed. Metrics belong in `data-pipeline/evaluation/search_evaluation.ipynb`.

In [1]:
import json
import unicodedata
from pathlib import Path

import chromadb
from sentence_transformers import SentenceTransformer

def find_project_root(start):
    for directory in (start, *start.parents):
        if (directory / 'data-pipeline').is_dir():
            return directory
    raise RuntimeError('Run this notebook inside the legal-rag project')

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / 'data-pipeline' / 'data'
CHROMA_DIR = DATA_DIR / 'processed' / 'chroma_db'
CHUNK_VARIANT = 'recursive-450-tokens'
COLLECTION_NAME = 'legal_chunks_bge_m3_recursive_450_tokens'
MODEL_NAME = 'BAAI/bge-m3'
CANDIDATE_COUNT = 100
TOP_K = 5

SPELLING_VARIANTS = {'សំលៀកបំពាក់': 'សម្លៀកបំពាក់', 'ចំរៀង': 'ចម្រៀង', 'ជ័យជម្នះ': 'ជ័យជំនះ'}
def normalize_query(query):
    query = unicodedata.normalize('NFD', str(query))
    clusters, current = [], ''
    for char in query:
        if unicodedata.combining(char) == 0 and current:
            clusters.append(current); current = ''
        current += char
    if current: clusters.append(current)
    query = ''.join(cluster[0] + ''.join(sorted(cluster[1:], key=unicodedata.combining)) if len(cluster) > 1 else cluster for cluster in clusters)
    query = unicodedata.normalize('NFC', query).replace('\u200b', '').replace('\u200c', '').replace('\u200d', '')
    for source, target in SPELLING_VARIANTS.items(): query = query.replace(source, target)
    return ' '.join(query.split())

articles = {}
for line in (DATA_DIR / 'processed' / 'articles_normalized.jsonl').read_text(encoding='utf-8').splitlines():
    if not line.strip():
        continue
    article = json.loads(line)
    article_id = str(article.get('article_id') or f"{article.get('document_id')}-{article.get('article_number')}")
    current = articles.get(article_id)
    if current is None or len(article.get('text', '')) > len(current.get('text', '')):
        articles[article_id] = article

client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = client.get_collection(COLLECTION_NAME)
model = SentenceTransformer(MODEL_NAME)
print('Collection:', COLLECTION_NAME, 'chunks:', collection.count(), 'articles:', len(articles))


W1103 00:49:39.680000 44761 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Collection: legal_chunks_bge_m3_450 chunks: 1193 articles: 468


In [2]:
def search(query, top_k=TOP_K, candidate_count=CANDIDATE_COUNT):
    query = normalize_query(query)
    vector = model.encode([query], normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)[0]
    result = collection.query(
        query_embeddings=[vector.tolist()],
        n_results=min(collection.count(), max(candidate_count, top_k * 20)),
        include=['documents', 'metadatas', 'distances'],
    )
    by_article = {}
    for index, chunk_id in enumerate(result['ids'][0]):
        metadata = result['metadatas'][0][index] or {}
        article_id = str(metadata.get('article_id') or metadata.get('parent_id') or chunk_id)
        match = {
            'chunk_id': chunk_id,
            'chunk_text': result['documents'][0][index] or '',
            'distance': float(result['distances'][0][index]),
        }
        by_article.setdefault(article_id, {'article_id': article_id, 'matches': []})['matches'].append(match)
    ranked = sorted(by_article.values(), key=lambda item: min(x['distance'] for x in item['matches']))
    output = []
    for rank, item in enumerate(ranked[:top_k], start=1):
        article = articles.get(item['article_id'], {})
        output.append({
            'rank': rank,
            'article_id': item['article_id'],
            'title': article.get('document_title'),
            'document_id': article.get('document_id'),
            'article_number': article.get('article_number'),
            'full_article': article.get('text', ''),
            'matched_chunks': sorted(item['matches'], key=lambda x: x['distance'])[:2],
            'source_url': article.get('source_url'),
            'pdf_url': article.get('pdf_url'),
        })
    return output


In [5]:
query = "ប្រសិនបើប្រទេសមួយដកខ្លួនចេញពីសន្ធិសញ្ញាស្តីពីច្បាប់ប្រកាសនីយបត្រតក្កកម្ម តើការដកខ្លួននោះប៉ះពាល់ដល់ពាក្យសុំប៉ាតង់ដែលកំពុងដំណើរការដែរឬទេ?"
results = search(query)
for result in results:
    print(f"\n{result['rank']}. {result['article_id']} — {result['title']}")
    print('Matched chunk:', result['matched_chunks'][0]['chunk_text'][:500])
    print('Full article:', result['full_article'][:1000])
    print('Source:', result['source_url'])


1. 11163-24 — សន្ធិសញ្ញា ស្តីពីច្បាប់ប្រកាសនីយបត្រតក្កកម្ម
Matched chunk: ការប្រកាសដកខ្លួនចេញពីសន្ធិសញ្ញា (១) [ ការជូនដំណឹង ] ភាគីហត្ថលេខីណាមួយ អាចប្រកាសដកខ្លួនចេញពីសន្ធិសញ្ញា ដោយជូនដំណឹងទៅអគ្គនាយក (២) [ កាលបរិច្ឆេទមានអានុភាព ] ការប្រកាសដកខ្លួនចេញណាមួយ ត្រូវមានអានុភាពមួយឆ្នាំបន្ទាប់ពីកាលបរិច្ឆេទដែលអគ្គនាយកបានទទួលសេចក្តីជូនដំណឹង ឬ តាមកាលបរិច្ឆេទបន្ទាប់ណាមួយដែលបង្ហាញក្នុងការជូនដំណឹង ការប្រកាសដកខ្លួនចេញ មិនត្រូវធ្វើឱ្យប៉ះពាល់ដល់ការអនុវត្តសន្ធិសញ្ញានេះចំពោះសំណុំលិខិតស្នើសុំដែលកំពុងព្យួរនៅឡើយ ឬ ចំពោះប្រកាសនីយបត្រកិច្ចព្រមព្រៀងដែលកំពុងមានអានុភាពពាក់ព័ន្ធនឹងការប្រកាសដកខ្លួនច
Full article: ការប្រកាសដកខ្លួនចេញពីសន្ធិសញ្ញា
(១) [ ការជូនដំណឹង ]
ភាគីហត្ថលេខីណាមួយ អាចប្រកាសដកខ្លួនចេញពីសន្ធិសញ្ញា ដោយជូនដំណឹងទៅអគ្គនាយក
(២) [ កាលបរិច្ឆេទមានអានុភាព ]
ការប្រកាសដកខ្លួនចេញណាមួយ ត្រូវមានអានុភាពមួយឆ្នាំបន្ទាប់ពីកាលបរិច្ឆេទដែលអគ្គនាយកបានទទួលសេចក្តីជូនដំណឹង ឬ តាមកាលបរិច្ឆេទបន្ទាប់ណាមួយដែលបង្ហាញក្នុងការជូនដំណឹង ការប្រកាសដកខ្លួនចេញ មិនត្រូវធ្វើឱ្យប៉ះពាល់ដល់ការអនុវត្តសន្ធិសញ្ញានេះចំពោះសំណុំលិខិតស្នើសុំដែលកំពុងព្